# Week 9 Live Coding
## The donor's portfolio

Four GOTV tactics. Each has a published effect and a cost. Which produces the most votes per dollar?

Five things we will do:
1. Compute cost per vote for each tactic
2. Fix the units on canvassing and phone, and watch \$1,000 turn into \$300
3. Compute total votes produced per \$500K
4. Show how uncertainty changes the ranking
5. Do the Bayesian update and re-price canvassing

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy. Work in that tab; edits to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup



Run the cell below to load the data.

In [ ]:
import pandas as pd

import numpy as np



tactics = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/'

                      'data_science_campaigns_26/main/weeks/wk09_cost_effectiveness/data/'

                      'gotv_tactics.csv')

tactics

## Part 1: Cost per additional vote

**cost per vote = cost / (effect size as a proportion)**

Mail costs \$0.60 a piece and moves turnout +0.7 pp, so \$0.60 / 0.007 = about \$86 a vote.

Watch the units! A percentage point is not a percent. The effect goes in as a proportion (+0.7 pp = 0.007), not as 0.7. This is a pet peeve of mine.

Let's run it on all four tactics the way the donor's table is written, and then look hard at what we get.

In [ ]:
# Cost per vote, dividing the donor's cost column by the effect column
tactics['cost_per_vote'] = tactics['cost_per_contact'] / (tactics['effect_pp'] / 100)
tactics[['tactic', 'effect_pp', 'cost_per_contact', 'cost_per_vote']]

**Stop and read the `cost_per_vote` column.** The ranking by effect (canvassing > mail > phone > texts) is not the ranking by cost per vote (texts, then mail, then phone, then canvassing).

The best tactic *per person* is not the best tactic *per dollar*. That is the headline of the week.

But one of those four numbers is wrong, and it is the biggest one.

## Part 2: Match the units

\$20 buys one canvassing **conversation**. But +2.0 pp is the effect of one **door knocked**, and most doors do not open. We divided a per-conversation cost by a per-door effect.

From Week 5: canvassers reach about **30%** of the doors they knock. So a door costs \$20 × 0.30 = \$6, not \$20. Phone has the same problem: \$2.00 buys a completed call, phone banks reach about **25%** of the numbers they dial, so a dial costs \$0.50.

The data has a `contact_rate` column for exactly this. Mail and texts are 1.00: you pay to send, and everyone on the list gets one.

In [ ]:
# Knock 100 doors, hold 30 conversations, pay 30 x $20 = $600, and spread that
# over all 100 doors: $6 each. Multiplying by the contact rate does exactly that.
tactics['cost_per_attempt'] = tactics['cost_per_contact'] * tactics['contact_rate']
tactics['cost_per_vote'] = tactics['cost_per_attempt'] / (tactics['effect_pp'] / 100)

tactics[['tactic', 'contact_rate', 'cost_per_contact', 'cost_per_attempt', 'cost_per_vote']]

Canvassing went from \$1,000 a vote to **\$300**, and phone from \$400 to **\$100**. Same tactics, same studies, off by a factor of 3.3 and 4 because the top and the bottom of the fraction described different people.

Now look at the whole column. The ranking by cost per vote is **texts, mail, phone, canvassing** — a swing from first to last in both directions: the biggest effect buys the most expensive vote and the smallest buys the cheapest. The tactic with the biggest effect per person buys the most expensive vote.

## Part 3: Total votes per \$500K

How many additional votes does each tactic produce with a \$500K budget?

In [ ]:
budget = 500_000

# How many people can you afford to try?
tactics['attempts'] = (budget / tactics['cost_per_attempt'])

# How many additional votes does that produce?
tactics['additional_votes'] = tactics['attempts'] * (tactics['effect_pp'] / 100)

tactics[['tactic', 'cost_per_attempt', 'attempts', 'effect_pp', 'additional_votes']].round(2)

Texts and mail each produce around 6,000 votes, phone 5,000, canvassing about 1,700.

This is the W1 lesson at work: **the low marginal cost of mail lets you scale.** Canvassing has a high marginal cost, which limits your reach. Note that fixing the units cut canvassing's disadvantage from nearly 12x to about 3.5x. It did not erase it.

## Part 4: Uncertainty changes the ranking

Every effect estimate has a confidence interval. Let's see what happens to cost per vote at the edges of the CI.

In [ ]:
# Cost per vote at the low and high end of the CI
# Note the inverse: a HIGHER effect means LOWER cost per vote.
tactics['cpv_low_ci'] = tactics['cost_per_attempt'] / (tactics['effect_high_pp'] / 100)
tactics['cpv_high_ci'] = tactics['cost_per_attempt'] / (tactics['effect_low_pp'] / 100)

# The texts CI includes 0, so its worst case is infinite
tactics.loc[tactics['effect_low_pp'] <= 0, 'cpv_high_ci'] = float('inf')

tactics[['tactic', 'cost_per_vote', 'cpv_low_ci', 'cpv_high_ci']].round(0)

Texts look great at the point estimate (\$80/vote), but the CI includes zero. If the true effect is zero, the cost per vote is infinite: you spend \$500,000 and buy nothing.

That is not a number you can average against \$80. It is a reason to hedge. A donor who cannot afford a total miss should prefer mail, whose interval excludes zero and whose worst case is a finite \$200.

## Part 5: A new canvassing study arrives

A 2024 field experiment finds +0.8 pp (CI [+0.2, +1.4]) in competitive state legislative races. Your prior, from the older literature, is +2.0 pp (CI [+1.0, +3.0]).

How much should the new study move you? **Bayesian updating** is a weighted average, and the weight on each estimate is its **precision**, which is 1 / SE².

You already have both standard errors. A 95% interval is the estimate ± 1.96 SE, so run it backwards: SE = (half the interval width) / 1.96.

Why squared and not just 1/SE? Take the square on faith this week. It is what makes the weighted average come out right, and it does change the answer.

In [ ]:
new_study = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/'
                        'data_science_campaigns_26/main/weeks/wk09_cost_effectiveness/data/'
                        'new_canvassing_study.csv')
print(new_study.to_string(index=False))

In [ ]:
# SE from a 95% interval: half the width, divided by 1.96
prior_est, prior_se = 2.0, (3.0 - 1.0) / (2 * 1.96)
new_est,   new_se   = 0.8, (1.4 - 0.2) / (2 * 1.96)

# Precision is 1 / SE^2. A tight estimate gets a big weight.
w_prior = 1 / prior_se**2
w_new   = 1 / new_se**2

updated = (w_prior * prior_est + w_new * new_est) / (w_prior + w_new)

print(f'prior:  +{prior_est} pp, SE {prior_se:.3f}, precision {w_prior:.2f}')
print(f'new:    +{new_est} pp, SE {new_se:.3f}, precision {w_new:.2f}')
print(f'\nweight on the prior: {w_prior/(w_prior+w_new):.0%}')
print(f'weight on the new study: {w_new/(w_prior+w_new):.0%}')
print(f'\nupdated belief: +{updated:.1f} pp')

**The prior gets 26% of the weight and the single new study gets 74%.** Dozens of experiments, outvoted by one.

Check it against the square: the new study's SE is 0.306 against the prior's 0.510, a ratio of 1.67, and 1.67² = 2.8. The study counts 2.8 times as heavily, exactly as if it rested on 2.8 times the data. "Dozens of experiments" was never the same thing as "precise."

Now run the three beliefs through the same \$6 a door and see what the donor's plan does.

In [ ]:
# Price the belief we would actually quote, rounded to one decimal like the slide
for label, effect in [('old literature only', 2.0), ('updated', round(updated, 1)), ('new study only', 0.8)]:
    print(f'{label:22s}  +{effect:.1f} pp  ->  ${6.00 / (effect/100):,.0f} per vote')

Updating moved canvassing from \$300 to about \$545 a vote. Mail is \$86, phone \$100, texts \$80.

Canvassing was already the most expensive vote on the table, so nothing reordered. What changed is the size of the gap: from 3.5x the price of mail to more than 6x. That is the difference between a tactic you fund because it buys volunteers and organization, and one you cannot defend on votes at all. One study, one afternoon of arithmetic.

---

## What you've seen today

- **Cost per vote = cost / effect.** Check that both describe the same people before you divide. Canvassing was off by 3.3x.
- **The ranking by effect is not the ranking by cost per vote.**
- **Bayesian updating is a weighted average, and the weights are precisions.** Twenty studies that disagree with each other are weak evidence.

**Lying-with-data tag #8:** Ranking tactics by effect size and never by cost. Dividing a cost and an effect that describe different people. Quoting a cost per vote without saying which vote.

Next, open `wk09_problem_set.ipynb`.